# 🎙️ Speech Intelligence Engine — Google Colab GPU Trainer

This notebook offloads heavy **Speech Foundation Model** training, SSL feature extraction (WavLM / HuBERT), and Speech-LLM adapter fine-tuning to Google Colab GPUs (T4 / L4 / A100), persisting all checkpoints directly to your 5TB **Google Drive** (`rakshithgoud453@gmail.com`).

## Step 1: Check GPU & Mount Google Drive (5TB Storage)

In [ ]:
# Check available GPU resources
!nvidia-smi

# Mount 5TB Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
DATASET_DIR = os.path.join(DRIVE_BASE_PATH, 'datasets')
OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, 'outputs')

for path in [DRIVE_BASE_PATH, CHECKPOINT_DIR, DATASET_DIR, OUTPUT_DIR]:
    os.makedirs(path, exist_ok=True)
    print(f"✓ Ready directory on Google Drive: {path}")

## Step 2: Install ML & Audio Dependencies (Colab-Optimized)

In [ ]:
# Install extra NLP, Audio & ML packages without overriding Colab's pre-installed CUDA PyTorch
!pip install -q transformers datasets peft accelerate bitsandbytes librosa soundfile wandb pyannote.audio faster-whisper hdbscan

## Step 3: Clone / Sync Repository Code

In [ ]:
# Codebase setup in Colab workspace
import os

REPO_URL = "https://github.com/rakshithgoud453/speech_intelligence_engine.git"
WORKSPACE_DIR = "/content/speech_intelligence_engine"

if os.path.exists(WORKSPACE_DIR):
    %cd {WORKSPACE_DIR}
    !git pull
else:
    !git clone {REPO_URL} {WORKSPACE_DIR}
    %cd {WORKSPACE_DIR}

print(f"✓ Currently operating in: {os.getcwd()}")

## Step 4: SSL Layer Representation Extraction (WavLM / HuBERT)

In [ ]:
import torch
import torchaudio
from transformers import WavLMModel, AutoFeatureExtractor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load Pre-trained SSL Model (microsoft/wavlm-base-plus)
MODEL_ID = "microsoft/wavlm-base-plus"
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
ssl_model = WavLMModel.from_pretrained(MODEL_ID, output_hidden_states=True).to(device)
ssl_model.eval()

def extract_ssl_representations(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)
    
    inputs = feature_extractor(waveform.squeeze(0).numpy(), sampling_rate=16000, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = ssl_model(**inputs)
        hidden_states = outputs.hidden_states
    
    # Layers 4-8: Prosodic/Phonetic representations
    prosodic_rep = torch.stack(hidden_states[4:9]).mean(dim=0)
    # Layers 9-11: Semantic representations
    semantic_rep = torch.stack(hidden_states[9:12]).mean(dim=0)
    
    return {
        "prosodic_embedding": prosodic_rep.cpu().numpy(),
        "semantic_embedding": semantic_rep.cpu().numpy()
    }

print("✓ SSL Feature Extraction Pipeline Loaded Successfully.")

## Step 5: Fine-Tuning & Saving Checkpoints to 5TB Google Drive

In [ ]:
import time

# Example checkpoint saver function
def save_checkpoint(model, step, metrics):
    checkpoint_name = f"speech_engine_step_{step}_{int(time.time())}.pt"
    save_path = os.path.join(CHECKPOINT_DIR, checkpoint_name)
    
    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'metrics': metrics
    }, save_path)
    
    print(f"✓ Saved Checkpoint to Google Drive (5TB): {save_path}")

print(f"✓ Checkpoints will be permanently stored in: {CHECKPOINT_DIR}")